# 01 - ETa and soil moisture rasters for the growing season

The daily 30 m ETa maps (Lamichhane et al., 2025a) and the soil moisture maps at six depths
(Lamichhane et al., 2025b) are averaged to monthly means, then warped onto the projection of the field DEM
so they can be sampled at the 5 m yield pixels in notebook 03.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import rioxarray as rxr
import xarray as xr
from osgeo import gdal

In [ ]:
RAW = Path("../data/raw")
DEM = RAW / "topography/DEM.tif"
OUT = Path("../data/processed/monthly")

# daily rasters with a YYYY-MM-DD date in the file name
SOURCES = {
    "ETa": RAW / "eta_daily",
    **{f"SM{d}": RAW / f"sm_daily/SM{d}" for d in (30, 60, 90, 120, 150, 180)},
}
RESOLUTION = 10    # m; nearest-neighbour, the values are read at the 5 m yield pixel centres later

## Monthly means

Only months with at least one image are written. The pixel grid is not changed at this step.

In [ ]:
def date_of(path):
    m = re.search(r"(\d{4}-\d{2}-\d{2})", path.name)
    return pd.to_datetime(m.group(1)) if m else None


def monthly_means(src_dir, var, out_dir):
    layers = []
    for tif in sorted(src_dir.glob("*.tif")):
        d = date_of(tif)
        if d is None:
            print(f"no date in {tif.name}, skipped")
            continue
        layers.append(rxr.open_rasterio(tif, masked=True).squeeze().expand_dims(time=[d]))
    stack = xr.concat(layers, dim="time").sortby("time")
    stack = stack.assign_coords(month=stack.time.dt.strftime("%Y-%m"))

    out_dir.mkdir(parents=True, exist_ok=True)
    for month, group in stack.groupby("month"):
        group.mean(dim="time").squeeze().rio.to_raster(out_dir / f"{var}_{month}.tif")
    return stack.month.to_series().nunique()


for var, src in SOURCES.items():
    n = monthly_means(src, var, OUT / "native" / var)
    print(f"{var}: {n} months")

## Align with the DEM

In [ ]:
dem_srs = gdal.Open(str(DEM)).GetProjection()
warp = gdal.WarpOptions(format="GTiff", xRes=RESOLUTION, yRes=RESOLUTION, dstSRS=dem_srs,
                        resampleAlg="near", targetAlignedPixels=True)

for var in SOURCES:
    out_dir = OUT / "aligned" / var
    out_dir.mkdir(parents=True, exist_ok=True)
    for tif in sorted((OUT / "native" / var).glob("*.tif")):
        gdal.Warp(str(out_dir / tif.name), str(tif), options=warp)
    print(f"{var}: aligned")